# Word2Vec Embeddings with Gensim

This notebook demonstrates how to train and inspect word embeddings using the Word2Vec implementation in Gensim.

It uses the local repository corpus:

```text
NLP/datasets/alice_in_wonderland.txt
```

No external download is required.

## Learning Objectives

After completing this notebook, students should be able to:

1. load and preprocess a local text corpus;
2. organize text as tokenized sentences;
3. train a Continuous Bag-of-Words Word2Vec model;
4. inspect vocabulary size and embedding dimensions;
5. retrieve words with similar vector representations;
6. calculate word-to-word similarity;
7. project embeddings into two dimensions with PCA and t-SNE;
8. explain the limitations of embeddings learned from a small corpus.

## 1. Import the required libraries

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

## 2. Load the local corpus

The notebook is stored in `NLP/notebooks`, so the dataset is referenced through a portable relative path.

In [ ]:
data_path = Path("../datasets/alice_in_wonderland.txt")

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {data_path.resolve()}"
    )

raw_text = data_path.read_text(
    encoding="utf-8",
    errors="replace",
)

print("Characters:", len(raw_text))
print("Preview:")
print(raw_text[:500])

## 3. Convert the text into tokenized sentences

Word2Vec expects an iterable of sentences, where each sentence is represented as a list of tokens.

This teaching example:

- converts text to lowercase;
- separates sentences using terminal punctuation;
- keeps alphabetic words and internal apostrophes;
- removes empty sentences.

In [ ]:
def tokenize_sentence(sentence):
    return re.findall(
        r"[a-z]+(?:'[a-z]+)?",
        sentence.lower(),
    )


sentence_strings = re.split(
    r"(?<=[.!?])\s+",
    raw_text,
)

corpus = [
    tokens
    for sentence in sentence_strings
    if (tokens := tokenize_sentence(sentence))
]

token_count = sum(len(sentence) for sentence in corpus)

print("Sentence count:", len(corpus))
print("Token count:", token_count)
print("First three tokenized sentences:")

for sentence in corpus[:3]:
    print(sentence)

## 4. Train a Continuous Bag-of-Words model

Word2Vec learns vectors from local word context.

The main parameters are:

- `vector_size`: number of dimensions in each embedding;
- `window`: maximum context distance around a target word;
- `min_count`: minimum corpus frequency required for inclusion;
- `sg=0`: selects Continuous Bag-of-Words rather than Skip-gram;
- `workers=1` and a fixed seed: improve reproducibility;
- `epochs`: number of passes through the corpus.

CBOW predicts a target word from surrounding context words.

In [ ]:
model = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=3,
    sg=0,
    workers=1,
    seed=42,
    epochs=30,
)

print("Vocabulary size:", len(model.wv))
print("Embedding dimension:", model.wv.vector_size)

## 5. Inspect the learned vocabulary and vectors

In [ ]:
vocabulary = list(model.wv.index_to_key)

print("First 20 vocabulary terms:")
print(vocabulary[:20])

example_word = "alice"

if example_word in model.wv:
    print(
        f"\nFirst 10 dimensions for {example_word!r}:"
    )
    print(model.wv[example_word][:10])
else:
    print(
        f"{example_word!r} is not present in the vocabulary."
    )

## 6. Retrieve similar words

Cosine similarity compares vector direction. Words that occur in related contexts may receive similar vectors.

In [ ]:
query_words = [
    word
    for word in ["alice", "queen", "rabbit", "king"]
    if word in model.wv
]

for word in query_words:
    print(f"Most similar to {word!r}:")
    for similar_word, score in model.wv.most_similar(
        word,
        topn=5,
    ):
        print(
            f"  {similar_word:<15} "
            f"{score:.4f}"
        )
    print()

## 7. Compare selected word pairs

In [ ]:
word_pairs = [
    ("king", "queen"),
    ("alice", "rabbit"),
    ("cat", "mouse"),
    ("day", "night"),
]

for first_word, second_word in word_pairs:
    if (
        first_word in model.wv
        and second_word in model.wv
    ):
        similarity = model.wv.similarity(
            first_word,
            second_word,
        )

        print(
            f"{first_word:<10} ↔ "
            f"{second_word:<10}: "
            f"{similarity:.4f}"
        )
    else:
        missing = [
            word
            for word in (first_word, second_word)
            if word not in model.wv
        ]

        print(
            f"Skipped {first_word!r}, {second_word!r}; "
            f"missing: {missing}"
        )

## 8. Select frequent words for visualization

Plotting the complete vocabulary can make labels unreadable and can make t-SNE unnecessarily expensive.

The following cells use only the most frequent words.

In [ ]:
plot_word_count = min(80, len(model.wv))

plot_words = model.wv.index_to_key[
    :plot_word_count
]

plot_vectors = model.wv[plot_words]

print("Words selected for plotting:", len(plot_words))

## 9. Visualize embeddings with PCA

Principal Component Analysis finds linear directions that explain the greatest variance in the embedding matrix.

The two-dimensional projection is useful for inspection, but it does not preserve every relationship from the original 100-dimensional space.

In [ ]:
pca = PCA(n_components=2)

pca_coordinates = pca.fit_transform(
    plot_vectors
)

plt.figure(figsize=(12, 9))
plt.scatter(
    pca_coordinates[:, 0],
    pca_coordinates[:, 1],
    s=25,
)

for word, (x_coordinate, y_coordinate) in zip(
    plot_words,
    pca_coordinates,
):
    plt.annotate(
        word,
        (x_coordinate, y_coordinate),
        fontsize=8,
        alpha=0.8,
    )

plt.title("Word2Vec Embeddings Projected with PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.tight_layout()
plt.show()

print(
    "Explained variance ratio:",
    pca.explained_variance_ratio_,
)

## 10. Visualize embeddings with t-SNE

t-SNE is a nonlinear projection method that emphasizes local neighborhoods.

Its axes do not have a direct semantic interpretation. Distances and clusters in the plot should be treated as exploratory rather than definitive.

The perplexity must be smaller than the number of plotted words, so it is selected dynamically.

In [ ]:
if len(plot_words) >= 5:
    perplexity = min(
        30,
        max(2, len(plot_words) // 4),
        len(plot_words) - 1,
    )

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
        random_state=42,
        max_iter=1_000,
    )

    tsne_coordinates = tsne.fit_transform(
        plot_vectors
    )

    tsne_points = pd.DataFrame(
        {
            "word": plot_words,
            "x": tsne_coordinates[:, 0],
            "y": tsne_coordinates[:, 1],
        }
    )

    plt.figure(figsize=(12, 9))
    plt.scatter(
        tsne_points["x"],
        tsne_points["y"],
        s=25,
    )

    for row in tsne_points.itertuples(
        index=False
    ):
        plt.annotate(
            row.word,
            (row.x, row.y),
            fontsize=8,
            alpha=0.8,
        )

    plt.title(
        "Word2Vec Embeddings Projected with t-SNE"
    )
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.tight_layout()
    plt.show()
else:
    print(
        "The vocabulary is too small for a useful "
        "t-SNE visualization."
    )

## Result Interpretation

Word2Vec does not assign meaning directly. It learns statistical associations from neighboring words.

Words may appear close together because they occur in similar contexts, not necessarily because they are strict synonyms.

For example:

- character names may cluster because they appear in similar narrative contexts;
- conversational words may cluster because they occur in dialogue;
- frequent function words may dominate parts of the embedding space;
- rare or ambiguous terms may receive unstable representations.

## PCA versus t-SNE

### PCA

- uses a linear projection;
- is deterministic for the same input;
- provides an explained-variance ratio;
- is useful for broad global structure.

### t-SNE

- uses a nonlinear projection;
- emphasizes local neighborhoods;
- depends on parameters such as perplexity;
- does not preserve all global distances;
- may show different layouts under different settings.

Neither two-dimensional plot fully represents the original embedding space.

## Limitations

The embeddings are trained on one relatively small literary corpus. Therefore:

- vocabulary coverage is limited;
- rare words have weak estimates;
- semantic relationships may be noisy;
- historical literary language affects the learned context;
- one vector is learned per word form, regardless of multiple meanings;
- punctuation and capitalization information are discarded;
- results should not be compared directly with large pretrained embeddings.

## Exercises

1. Compare CBOW with Skip-gram by setting `sg=1`.
2. Change the context window.
3. Compare different embedding dimensions.
4. Increase or decrease `min_count`.
5. inspect the most frequent words.
6. Compare results across random seeds.
7. Save and reload the model.
8. Train on another local corpus.
9. Compare Word2Vec with FastText.
10. Evaluate word analogies where the vocabulary supports them.